## Primary Key Decision: Simple 7-Column Business Key

### The Final Key

```sql
CONCAT_WS('_',
  FL_DATE, -- flight date
  AIRLINE_CODE, -- airline operator
  FL_NUMBER, -- flight number
  CRS_DEP_TIME, -- Scheduled departure time
  OriginAirportID, 
  DestAirportID,
  Tail_Number -- Physical aircraft ID 
) AS flight_id
```

**Result (Cell 14):** 52,864,767 rows → 52,864,767 distinct keys → **0 duplicates** ✓

**Example key:** `AA_1480_20260101_800_12892_10721_N12345`

---

## Key EDA Insights That Drove This Decision

### 1. Flight Numbers Are Reused Across Routes (Cell 8)
**Test:** `(FL_DATE, AIRLINE_CODE, FL_NUMBER, CRS_DEP_TIME)`  
**Result:** **744 duplicates (Mostly rare)** 

**Why:** Diverted flights or same flight number operating on different routes simultaneously
* Example: AA 1480 at 08:00 departs from both LAX→BOS AND ORD→DFW on same day
* Airlines reuse popular flight numbers across their network

**Implication:** Must include route (`OriginAirportID`, `DestAirportID`) in primary key

---

### 2. Same Aircraft Operates Multiple Flights Per Day (Cells 9, 12, 13)
**Test:** `(FL_DATE, AIRLINE_CODE, Tail_Number, CRS_DEP_TIME)`  
**Finding:** Physical Tail_Number alone is insufficient as primary key

**Cell 12 observation:** Same physical aircraft flies same route multiple times daily
* Example: N8329B operates LAX↔SFO shuttle at 08:00, 12:00, 16:00, 20:00
* High-frequency routes (shuttle operations) reuse same aircraft

**Cell 13 observation:** Same tail can have same departure time on different routes
* Example: N201JQ departs at 06:00 from both ORD→ATL and DFW→LAX
* Aircraft repositioning and multi-leg operations create this pattern

**Implication:** Cannot use Tail_Number as sole identifier; must combine with flight number OR route

---

### 3. Rare Aircraft Swaps (Cell 11)
**Test:** `(FL_DATE, AIRLINE_CODE, FL_NUMBER, CRS_DEP_TIME, OriginAirportID, DestAirportID)`  
**Result:** **1 duplicate** (YX 3624, June 21, 2018)

**What happened:**
* **Original flight:** N643RW departed 20:27 (20 min late)
* **Substitute aircraft:** N864RW deployed next morning 10:44 (877 min weather delay)
* Same flight number, date, scheduled time, and route — **different aircraft**

**Interpretation:** In some rare instances airlines can deploy a substitute aircraft with same flight identity (number) but different tail number

**Implication:** Adding `Tail_Number` resolves this final collision (1 duplicate → 0 duplicates)

---

### 4. NULL Tail Numbers Are Cancelled Flights (Cell 6)
**Finding:** 289,435 NULL Tail_Number rows (0.55% of dataset)
* 289,433 are cancelled (99.999%)
* 2 non-cancelled (0.00%)

**Business Logic:** No physical aircraft assigned due to early cancellation

**Implication:** NULL handling doesn't break uniqueness
* `CONCAT_WS` treats NULL as empty string
* The 6-column combination is already distinct for cancelled flights
* Business pattern prevents collision: airlines don't schedule duplicate flights on same route at same time

---

## Important Notes to Remember

### Why Each Column Is Essential

| Column | Purpose | Validated In |
|--------|---------|-------------|
| `FL_DATE` | Flight date | Implicit |
| `AIRLINE_CODE` | Carrier (AA, WN, DL...) | Implicit |
| `FL_NUMBER` | Flight number | Cell 8 |
| `CRS_DEP_TIME` | **Scheduled** departure time (handles shuttle operations) | Cell 8, 12 |
| `OriginAirportID` | Origin airport (same flight # serves different routes) | Cell 8 (744 duplicates without this) |
| `DestAirportID` | Destination airport (completes route definition) | Cell 8 (744 duplicates without this) |
| `Tail_Number` | Aircraft registration (resolves split operations) | Cell 11 (1 duplicate without this) |

### Uniqueness Progression
* Without route: **744 duplicates**
* With route, without tail: **1 duplicate** (YX 3624 weather recovery)
* With route + tail: **0 duplicates** ✓

### NULL Handling
* `CONCAT_WS` automatically handles NULLs by treating them as empty strings
* No COALESCE or fallback logic needed
* Example cancelled flight key: `WN_5678_20240315_1730_12892_10721_` (trailing underscore = NULL tail)

### Design Philosophy
* **Kimball's approach:** Facts sharing the same grain must remain the same physical table.
* **Human-readable:** Keys contain actual business identifiers (no surrogate integers)
* **dbt-friendly:** Works with `unique_key='flight_id'` incremental merge
* **Debuggable:** Can parse carrier, date, route directly from key string

### dbt Implementation Template

```sql
-- skyops/models/staging/stg_bts_flights.sql
WITH source AS (
  SELECT * FROM {{ source('bronze', 'bts_ontime') }}
)

SELECT
  -- Primary key
  CONCAT_WS('_',
    FL_DATE,
    AIRLINE_CODE,
    CAST(FL_NUMBER AS STRING),
    CRS_DEP_TIME,
    CAST(OriginAirportID AS STRING),
    CAST(DestAirportID AS STRING),
    Tail_Number
  ) AS flight_id,
  
  -- All source columns
  *
FROM source
```

### Validation Checklist

- [x] Cell 8: Verified route essential (744 duplicates without it) ✓
- [x] Cell 9: Verified tail number patterns ✓
- [x] Cell 11: Analyzed edge case (YX 3624 split operation) ✓
- [x] Cell 12: Verified shuttle operations (same tail, multiple times) ✓
- [x] Cell 13: Verified tail reuse across routes ✓
- [x] Cell 14: Verified uniqueness (52.8M rows → 0 duplicates) ✓
- [x] Cell 6: Confirmed NULL Tail_Number = 99.999% cancelled ✓
- [ ] Deploy to `skyops.silver.stg_bts_flights`
- [ ] Deploy to `skyops.gold.fct_flights` with `unique_key='flight_id'`
- [ ] Add dbt tests: `unique` and `not_null` on `flight_id`

In [0]:
%python
# Core EDA, identify the bronze data and what needs to be transformed for required analytics.

In [0]:
%sql
USE skyops.bronze

In [0]:
%python
from pyspark.sql import functions as f

In [0]:
%python
df = spark.table('skyops.bronze.bts_ontime')
df.describe().display()

In [0]:
-- tail number is mostly only null WHEN flight is cancelled -> no physical aircraft assigned to the ride due to early cancellation.
select count(*) from bts_ontime where Tail_Number is null and CANCELLED = 1.0

In [0]:
-- only 1 record with null flight record. handle it
SELECT * FROM bts_ontime WHERE year = 2024 IS NULL 

In [0]:
-- Observation 1: 744 instances of the same flight of an airline departing to/from multiple airports at the same time 
-- Reasons: Diverted flights.. eg. same plane can have the same departure time but in a different route if diverted, so first record shows diverted flag and next record may show the final updated route. 
SELECT
 FL_DATE, AIRLINE_CODE, FL_NUMBER, CRS_DEP_TIME,  COUNT(*)
FROM bts_ontime GROUP BY FL_DATE, AIRLINE_CODE, FL_NUMBER, CRS_DEP_TIME 
HAVING 
COUNT(*) != 1

In [0]:
-- Observation 1: 744 instances of the same flight of an airline departing from multiple airports at the same time 
-- Reasons: Diverted flights.. same plane can have the same departure time but in a different route if diverted, so first record shows diverted and next record may show the final updated route. 
SELECT
 FL_DATE, AIRLINE_CODE, tail_number, CRS_DEP_TIME,   COUNT(*)
FROM bts_ontime 
where tail_number is not null
GROUP BY FL_DATE, AIRLINE_CODE, tail_number, CRS_DEP_TIME 
HAVING 
COUNT(*) != 1

In [0]:
-- Final grain
SELECT
 FL_DATE, AIRLINE_CODE, FL_NUMBER, CRS_DEP_TIME, DestAirportID, OriginAirportID, tail_number, COUNT(*)
FROM bts_ontime GROUP BY FL_DATE, AIRLINE_CODE, FL_NUMBER, CRS_DEP_TIME, OriginAirportID, DestAirportID, tail_number 
HAVING 
COUNT(*) != 1

In [0]:
-- Observation 2: Rarely same flight number can have also same origin/destination/Departure time multiple times (but tail number would be different as the PHYSICAL PLANE for that flight would HAVE to be different). Airlines avoid this but for any sudden emergencies, this can happen.
select * from bts_ontime where FL_DATE = 20180621 and AIRLINE_CODE = 'YX' and FL_NUMBER = 3624 and CRS_DEP_TIME = 2007 

In [0]:
-- Observation 3: there can be same physical plane with same origin and dest airports in the same day ofcourse! 
SELECT
 FL_DATE, AIRLINE_CODE, Tail_Number, OriginAirportID, DestAirportID, COUNT(*)
FROM bts_ontime GROUP BY FL_DATE, AIRLINE_CODE, Tail_Number, OriginAirportID, DestAirportID HAVING Tail_Number IS NOT NULL AND COUNT(*) != 1

In [0]:
-- Observation 4: diff flights totally can exist with same tail number and same dep time
SELECT
 *
FROM bts_ontime 
WHERE FL_DATE = 20260224 AND Tail_Number = 'N201JQ' AND CRS_DEP_TIME = 600 AND AIRLINE_CODE = 'YX'

In [0]:
-- (FL_DATE, AIRLINE_CODE, FL_NUMBER, CRS_DEP_TIME, OriginAirportID, DestAirportID, Tail_Number)
-- WITHOUT any COALESCE or fallback logic

SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT 
    CONCAT_WS('|', 
      FL_DATE, 
      AIRLINE_CODE, 
      FL_NUMBER,  -- Can be NULL
      CRS_DEP_TIME, 
      OriginAirportID, 
      DestAirportID,
      Tail_Number  -- Can be NULL
    )
  ) AS distinct_keys,
  COUNT(*) - COUNT(DISTINCT 
    CONCAT_WS('|', 
      FL_DATE, 
      AIRLINE_CODE, 
      FL_NUMBER, 
      CRS_DEP_TIME, 
      OriginAirportID, 
      DestAirportID,
      Tail_Number
    )
  ) AS duplicates
FROM skyops.bronze.bts_ontime